SETUP DO AMBIENTE


In [ ]:
!pip install -U langchain langchain-community langchain-openai openai sqlalchemy

CONFIGURAÇÕES E IMPORT


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

GERANDO OS DADOS (ALEATÓRIO)

In [ ]:
def gerar_dados_dashboard(total_alunos=1000, taxa_anomalia=0.05):
    """
    total_alunos: quantidade de alunos na base.
    taxa_anomalia: percentual de alunos que terão erro (0.05 = 5%).
    """
    np.random.seed(42)
    alunos_ids = [f"MATRICULA_{i:04d}" for i in range(1, total_alunos + 1)]

    qtd_com_erro = int(total_alunos * taxa_anomalia)
    alunos_com_erro = np.random.choice(alunos_ids, size=qtd_com_erro, replace=False)

    registros = []
    for aluno in alunos_ids:

        base_valor = np.random.uniform(800, 2000)
        v_bol_29 = base_valor
        v_rob_29 = base_valor * 1.5
        v_bolsa_29 = base_valor * 0.15

        registros.append([aluno, '2025-05-29', v_bol_29, v_rob_29, v_bolsa_29])

        if aluno in alunos_com_erro:

            fator = np.random.choice([0.80, 1.25, 0.70, 1.30])
            v_bol_30 = v_bol_29 * fator
            v_rob_30 = v_rob_29 * fator
        else:

            v_bol_30 = v_bol_29 * np.random.uniform(0.99, 1.01)
            v_rob_30 = v_rob_29 * np.random.uniform(0.99, 1.01)

        registros.append([aluno, '2025-05-30', v_bol_30, v_rob_30, v_bolsa_29])

    df = pd.DataFrame(registros, columns=['id_aluno', 'data', 'boleto', 'rob', 'bolsa'])



    return df.dropna()


APLICANDO REGRAS DE NEGÓCIO

In [ ]:
def exibir_dashboard(df, data_hoje='2025-05-30', limite_top=10, limiar_anomalia=10):

    df = df.sort_values(['id_aluno', 'data'])


    df['dif_perc_boleto'] = df.groupby('id_aluno')['boleto'].pct_change() * 100
    df['dif_perc_rob'] = df.groupby('id_aluno')['rob'].pct_change() * 100

    hoje = df[df['data'] == data_hoje].copy()

    anomalias = hoje[
        (hoje['dif_perc_boleto'].abs() > limiar_anomalia) |
        (hoje['dif_perc_rob'].abs() > limiar_anomalia)
    ].copy()

    anomalias['maior_desvio'] = anomalias[['dif_perc_boleto', 'dif_perc_rob']].abs().max(axis=1)

    RED, GREEN, CYAN, RESET = "\033[91m", "\033[92m", "\033[96m", "\033[0m"

    print("\n" + "="*75)
    print(f"{CYAN}           DASHBOARD DE INTEGRIDADE DE DADOS - PROCESSAMENTO AI{RESET}")
    print("="*75)

    status = f"{RED}CRÍTICO{RESET}" if len(anomalias) > 0 else f"{GREEN}SAUDÁVEL{RESET}"
    print(f" STATUS DO BANCO: {status}")
    print(f" TOTAL DE ALUNOS MONITORADOS: {len(hoje)}")
    print(f" ANOMALIAS DETECTADAS: {len(anomalias)}")
    print("-" * 75)

    if not anomalias.empty:
        print(f"{RED}TOP {limite_top} CASOS PARA INTERVENÇÃO IMEDIATA (Ordenado por Gravidade):{RESET}")
        print(f"{'ID ALUNO':<15} | {'VAR BOLETO':<12} | {'VAR ROB':<12} | {'GRAVIDADE'}")
        print("-" * 75)

        anomalias = anomalias.sort_values(by='maior_desvio', ascending=False)

        for _, row in anomalias.head(limite_top).iterrows():
            gravidade = "ALTA" if row['maior_desvio'] > 15 else "MÉDIA"
            print(f"{row['id_aluno']:<15} | {row['dif_perc_boleto']:>10.1f}% | {row['dif_perc_rob']:>10.1f}% | {gravidade}")

        if len(anomalias) - limite_top > 0:
            print(f"\n... e mais {len(anomalias) - limite_top} registros ocultos. Verifique o log completo.")
    else:
        print(f"{GREEN}✓ Nenhum desvio identificado entre os {len(hoje)} alunos.{RESET}")

    print("="*75 + "\n")

GERANDO BASE E EXIBINDO DASHBOARD

In [ ]:
base_auditoria = gerar_dados_dashboard(total_alunos=1000, taxa_anomalia=0.03)


exibir_dashboard(base_auditoria, data_hoje='2025-05-30', limite_top=10, limiar_anomalia=10)


print("Visualização das colunas processadas (Head):")
print(base_auditoria.tail())


           DASHBOARD DE INTEGRIDADE DE DADOS - PROCESSAMENTO AI
 STATUS DO BANCO: CRÍTICO
 TOTAL DE ALUNOS MONITORADOS: 1000
 ANOMALIAS DETECTADAS: 30
---------------------------------------------------------------------------
TOP 10 CASOS PARA INTERVENÇÃO IMEDIATA (Ordenado por Gravidade):
ID ALUNO        | VAR BOLETO   | VAR ROB      | GRAVIDADE
---------------------------------------------------------------------------
MATRICULA_0903  |      -30.0% |      -30.0% | ALTA
MATRICULA_0102  |       30.0% |       30.0% | ALTA
MATRICULA_0236  |      -30.0% |      -30.0% | ALTA
MATRICULA_0528  |      -30.0% |      -30.0% | ALTA
MATRICULA_0372  |       30.0% |       30.0% | ALTA
MATRICULA_0175  |      -30.0% |      -30.0% | ALTA
MATRICULA_0522  |      -30.0% |      -30.0% | ALTA
MATRICULA_0860  |       30.0% |       30.0% | ALTA
MATRICULA_0884  |       30.0% |       30.0% | ALTA
MATRICULA_0812  |      -30.0% |      -30.0% | ALTA

... e mais 20 registros ocultos. Verifique o log completo.

Vi

GRAVANDO NO SQLITE

In [ ]:
DB_PATH = "auditoria.db"
TABLE_NAME = "auditoria_pagamentos"

conn = sqlite3.connect(DB_PATH)
base_auditoria.to_sql(TABLE_NAME, conn, if_exists="replace", index=False)


cur = conn.cursor()
cur.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
print("Total de registros na tabela:", cur.fetchone()[0])

cur.execute(f"PRAGMA table_info({TABLE_NAME})")
print("Schema (colunas):")
for row in cur.fetchall():
    print(row)

conn.close()

Total de registros na tabela: 2000
Schema (colunas):
(0, 'id_aluno', 'TEXT', 0, None, 0)
(1, 'data', 'TEXT', 0, None, 0)
(2, 'boleto', 'REAL', 0, None, 0)
(3, 'rob', 'REAL', 0, None, 0)
(4, 'bolsa', 'REAL', 0, None, 0)


REALIZANDO AS CONSULTAS

In [ ]:
conn = sqlite3.connect(DB_PATH)


query_dados_brutos = f"""
SELECT id_aluno, data, boleto, rob
FROM {TABLE_NAME}
WHERE data IN ('2025-05-29', '2025-05-30')
ORDER BY id_aluno, data;
"""

df_bruto = pd.read_sql_query(query_dados_brutos, conn)
conn.close()


df_bruto['dif_perc_boleto'] = df_bruto.groupby('id_aluno')['boleto'].pct_change() * 100
df_bruto['dif_perc_rob'] = df_bruto.groupby('id_aluno')['rob'].pct_change() * 100

df_top10 = df_bruto[df_bruto['data'] == '2025-05-30'].copy()
df_top10['maior_desvio'] = df_top10[['dif_perc_boleto', 'dif_perc_rob']].abs().max(axis=1)
df_top10 = df_top10[df_top10['maior_desvio'] > 10].sort_values('maior_desvio', ascending=False).head(10)

print(df_top10)

            id_aluno        data       boleto          rob  dif_perc_boleto  dif_perc_rob  maior_desvio
1805  MATRICULA_0903  2025-05-30  1212.504006  1818.756009            -30.0         -30.0          30.0
203   MATRICULA_0102  2025-05-30  2034.081264  3051.121897             30.0          30.0          30.0
471   MATRICULA_0236  2025-05-30  1068.652052  1602.978079            -30.0         -30.0          30.0
1055  MATRICULA_0528  2025-05-30   759.444113  1139.166170            -30.0         -30.0          30.0
743   MATRICULA_0372  2025-05-30  2071.738153  3107.607230             30.0          30.0          30.0
349   MATRICULA_0175  2025-05-30  1174.266856  1761.400284            -30.0         -30.0          30.0
1043  MATRICULA_0522  2025-05-30   587.422369   881.133554            -30.0         -30.0          30.0
1719  MATRICULA_0860  2025-05-30  2025.013517  3037.520276             30.0          30.0          30.0
1767  MATRICULA_0884  2025-05-30  1201.738586  1802.607879      

CONFIGURAÇÃO OPENAI API KEY

In [ ]:
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Cole sua OPENAI_API_KEY: ")

Cole sua OPENAI_API_KEY: ··········


APLICANDO LANGCHAIN

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

agent = create_sql_agent(
    llm=llm,
    db=db,
    verbose=True
)

REALIZANDO 1° PERGUNTA

In [ ]:
agent.invoke({"input": f"Quantos registros existem na tabela {TABLE_NAME}?"})



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: 
auditoria_pagamentosQuestion: Quantos registros existem na tabela auditoria_pagamentos?
Thought: I know the table auditoria_pagamentos exists. I need to count the number of records in this table.
Action: sql_db_query_checker
Action Input: SELECT COUNT(*) FROM auditoria_pagamentos;SELECT COUNT(*) FROM auditoria_pagamentos;Action: sql_db_query
Action Input: SELECT COUNT(*) FROM auditoria_pagamentos;[(2000,)]Final Answer: A tabela auditoria_pagamentos possui 2000 registros.

> Finished chain.


{'input': 'Quantos registros existem na tabela auditoria_pagamentos?',
 'output': 'A tabela auditoria_pagamentos possui 2000 registros.'}

2° PERGUNTA

In [ ]:
agent.invoke({
    "input": f"""
Analise a tabela {TABLE_NAME}.
Como as variações percentuais não estão pré-calculadas, você deve:
1. Comparar os valores de 'boleto' e 'rob' do dia '2025-05-30' com o dia '2025-05-29' para cada aluno.
2. Calcular a variação percentual para cada um: ((Valor_dia_30 / Valor_dia_29) - 1) * 100.
3. Identificar o 'maior_desvio' como o valor máximo absoluto entre essas duas variações.
4. Listar os 10 alunos com os maiores desvios no dia 2025-05-30.

Retorne uma tabela com: id_aluno, dif_perc_boleto, dif_perc_rob e maior_desvio.
"""
})



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: auditoria_pagamentosThought: There is only one table named auditoria_pagamentos, which is the one I need to analyze. I will check the schema of this table to understand the column names and data types, especially to confirm the columns id_aluno, boleto, rob, and the date column.
Action: sql_db_schema
Action Input: auditoria_pagamentos
CREATE TABLE auditoria_pagamentos (
	id_aluno TEXT, 
	data TEXT, 
	boleto REAL, 
	rob REAL, 
	bolsa REAL
)

/*
3 rows from auditoria_pagamentos table:
id_aluno	data	boleto	rob	bolsa
MATRICULA_0001	2025-05-29	1987.9522788679342	2981.9284183019013	298.1928418301901
MATRICULA_0001	2025-05-30	1980.8892372936416	3000.408886625891	298.1928418301901
MATRICULA_0002	2025-05-29	1105.5687857165167	1658.353178574775	165.8353178574775
*/Thought: I need to compare the 'boleto' and 'rob' values for each 'id_aluno' between the dates '2025-05-29' and '2025-05-30'. Then calculate the perc

{'input': "\nAnalise a tabela auditoria_pagamentos. \nComo as variações percentuais não estão pré-calculadas, você deve:\n1. Comparar os valores de 'boleto' e 'rob' do dia '2025-05-30' com o dia '2025-05-29' para cada aluno.\n2. Calcular a variação percentual para cada um: ((Valor_dia_30 / Valor_dia_29) - 1) * 100.\n3. Identificar o 'maior_desvio' como o valor máximo absoluto entre essas duas variações.\n4. Listar os 10 alunos com os maiores desvios no dia 2025-05-30.\n\nRetorne uma tabela com: id_aluno, dif_perc_boleto, dif_perc_rob e maior_desvio.\n",
 'output': "A tabela com os 10 alunos que tiveram os maiores desvios percentuais entre os dias 2025-05-29 e 2025-05-30, considerando as variações percentuais de 'boleto' e 'rob', é:\n\n| id_aluno       | dif_perc_boleto | dif_perc_rob | maior_desvio |\n|----------------|-----------------|--------------|--------------|\n| MATRICULA_0903 | -30.00          | -30.00       | 30.00        |\n| MATRICULA_0102 | 30.00           | 30.00        |

FAZENDO A SELEÇÃO PARA APENAS O SELECT / CRIANDO REGRAS PARA O AGENTE


In [ ]:
import re

def somente_select(sql: str) -> bool:

    s = sql.strip().lower()

    proibidas = ["insert", "update", "delete", "drop", "alter", "create", "truncate", "attach", "pragma"]
    if any(p in s for p in proibidas):
        return False

    return s.startswith("select") or s.startswith("with")


teste_sql_ok = "SELECT * FROM auditoria_pagamentos LIMIT 5;"
teste_sql_bad = "DROP TABLE auditoria_pagamentos;"

print("OK?", somente_select(teste_sql_ok))
print("OK?", somente_select(teste_sql_bad))

OK? True
OK? False


SELECIONANDO AS PERGUNTAS QUE SERÃO FEITAS PARA ANALISE

In [ ]:

contexto_calculo = "Calcule a variação percentual comparando os valores do dia 2025-05-30 com o dia 2025-05-29."

perguntas = [
    f"{contexto_calculo} Quantas anomalias (variação de boleto ou rob > 10%) ocorreram na tabela {TABLE_NAME}?",

    f"{contexto_calculo} Qual a média e o máximo da variação absoluta de boleto e rob no dia 2025-05-30?",

    f"{contexto_calculo} Quais alunos tiveram desvio (boleto ou rob) acima de 15% (crítico)? Liste os 20 maiores IDs e suas variações.",

    f"{contexto_calculo} Entre a variação de boleto e a de rob, qual teve mais ocorrências acima de 10%?",

    f"{contexto_calculo} Quais são os 10 alunos com menor variação absoluta (mais próximos de 0%) no dia 2025-05-30?"
]

for p in perguntas:
    print("\n" + "="*80)
    print("PERGUNTA:", p)
    print("="*80)

    # O invoke agora leva o contexto do cálculo embutido na pergunta
    try:
        resp = agent.invoke({"input": p})
        # Dependendo da versão do LangChain, a resposta pode vir em 'output' ou ser o dicionário todo
        print(resp.get("output", resp) if isinstance(resp, dict) else resp)
    except Exception as e:
        print(f"Erro ao processar a pergunta: {e}")


PERGUNTA: Calcule a variação percentual comparando os valores do dia 2025-05-30 com o dia 2025-05-29. Quantas anomalias (variação de boleto ou rob > 10%) ocorreram na tabela auditoria_pagamentos?


> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: auditoria_pagamentosThought: There is only one table named auditoria_pagamentos, which is likely the relevant table for this question. I should check the schema of this table to understand the columns and find the columns related to date, boleto, rob, and values to calculate the percentage variation.

Action: sql_db_schema
Action Input: auditoria_pagamentos
CREATE TABLE auditoria_pagamentos (
	id_aluno TEXT, 
	data TEXT, 
	boleto REAL, 
	rob REAL, 
	bolsa REAL
)

/*
3 rows from auditoria_pagamentos table:
id_aluno	data	boleto	rob	bolsa
MATRICULA_0001	2025-05-29	1987.9522788679342	2981.9284183019013	298.1928418301901
MATRICULA_0001	2025-05-30	1980.8892372936416	3000.408886625891	298.1928418301901
MATRICULA_00